In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import sklearn
import sklearn.datasets
import sklearn.ensemble
import lime
import lime.lime_tabular
from __future__ import print_function

# System Call Log Creation:

In [2]:
import csv
# Write header
#writer.writerow(["Itr_Num", "Layer_Name", "Process_Name", "Log_Prompt"])
global Track_log
Track_log=1
def get_last_track_log(csv_filename):
    try:
        with open(csv_filename, mode="r", newline="") as file:
            reader = csv.reader(file)
            rows = list(reader)
            if rows:  # Ensure the file is not empty
                last_row = rows[-1]  # Get the last row
                return int(last_row[-1])  # Extract the last column (Track_log)
            else:
                return 1  # Default value if file is empty
    except FileNotFoundError:
        return 1  # Default if file doesn't exist
    

def Call_log( Layer_Name, Process_Name, Log_Prompt):
    # Define the CSV file name
    csv_filename = "System_Call_Log.csv"
    # Write data to CSV file
    global Track_log
    Track_log=get_last_track_log(csv_filename)
    itr_num=Track_log
    with open(csv_filename, mode="a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([itr_num, Layer_Name, Process_Name, Log_Prompt,Track_log])
        

    

# Read sensor data
PI collects the data and sends it to the fog layer through UDP connection.
Fog Layer is another Raspberry pi which collects all the information from the physical layer.
we assume data will be something like in the below csv file

In [3]:

df = pd.read_csv('./Soil_Temp_measure.csv', low_memory=False) 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 21 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   DateTime                                                    610 non-null    object 
 1   Soil temperture at 2 inch below the soil surface (F)        610 non-null    float64
 2   Soil data under the cover right on top of soil surface (F)  610 non-null    float64
 3   Air temperature 1.5m (F)                                    610 non-null    float64
 4   Dewpoint                                                    610 non-null    float64
 5   Wind Chill                                                  427 non-null    float64
 6   Wind Gust 10m                                               610 non-null    float64
 7   Wind Speed 10m                                              610 non-null    float64
 8   

# Clean sensor data
Clean them if necessary. here we have rewrite the columns name slightly.

In [4]:
#To replace white spaces with other characters (underscore for instance):
df.columns = df.columns.str.strip()#To remove white space at both ends:
df.columns = df.columns.str.replace(' ', '_')
df[['Cover_or_not']] = df[['Cover_or_not']].replace(' ', '', regex=True)
df.info()
#Dividing data into X and Y
feat_cols = list(df.columns)
label_col = "Cover_or_not"
feat_cols.remove(label_col)
X = df.drop([label_col], axis=1)
y = df[label_col]

feat_cols = list(X.columns)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 21 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   DateTime                                                    610 non-null    object 
 1   Soil_temperture_at_2_inch_below_the_soil_surface_(F)        610 non-null    float64
 2   Soil_data_under_the_cover_right_on_top_of_soil_surface_(F)  610 non-null    float64
 3   Air_temperature_1.5m_(F)                                    610 non-null    float64
 4   Dewpoint                                                    610 non-null    float64
 5   Wind_Chill                                                  427 non-null    float64
 6   Wind_Gust_10m                                               610 non-null    float64
 7   Wind_Speed_10m                                              610 non-null    float64
 8   

# SendSensorDataToEdge
after receive data from sensor, we need to send it to edge layer .we have used udp protocol  to send the data stream

In [5]:
import socket
import threading
import json
import random
import time
import pandas as pd

# Function to send data via UDP to the edge server
def SendSensorDataToEdge(data, ip, port):
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    message = json.dumps(data)
    client_socket.sendto(message.encode('utf-8'), (ip, port))
    print(f"Sent data to edge server via UDP: {data}")
    client_socket.close()


# TriggerActuatorToActivate
If Cloud analysis needs to activate some trigger on physical level,it send back to Edge layer.Edge layer then send back to physical
layer and this process handles receiving the command .It uses tcp connection and listen to the port for edge layer

In [6]:
# Function to listen for modified data from the edge server via TCP
def TriggerActuatorToActivate(tcp_ip, tcp_port):
    tcp_listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    try:
        tcp_listener.bind((tcp_ip, tcp_port))
        tcp_listener.listen(1)
        print(f"Physical server listening for modified data from edge server on {tcp_ip}:{tcp_port}")
        
        while True:
            conn, addr = tcp_listener.accept()
            print(f"Connected to edge server at {addr}")
            data_from_edge = conn.recv(1024).decode('utf-8')
            print(f"Data received from edge server via TCP: {data_from_edge}")
            #Call_log( "Physical Layer", "TriggerActuatorToActivate", "Activate Trigger")
            csv_filename = "System_Call_Log.csv"
            Track_log=get_last_track_log(csv_filename)
            itr_num=Track_log
            Track_log=int(Track_log)+1
            with open(csv_filename, mode="a+", newline="") as file:
                writer = csv.writer(file)
                writer.writerow([itr_num,"Physical Layer", "TriggerActuatorToActivate", "Activate Trigger",Track_log])

            conn.close()

    except Exception as e:
        print(f"Error while binding or listening on {tcp_port}: {e}")
    finally:
        tcp_listener.close()

# Main function


In [ ]:
# Main function to start UDP client and TCP listener
def main():
    # Sample DataFrame X (replace with your actual DataFrame)
    

    # Edge server's UDP details
    udp_ip = '127.0.0.1'  # Edge server IP
    udp_port = 12345       # Edge server's UDP port

    # Physical server's TCP details (to receive modified data)
    tcp_ip = '127.0.0.1'  # Physical server IP
    tcp_port = 12348      # Physical server's TCP port for receiving data

    # Start TCP listener for modified data in a separate thread
    tcp_thread = threading.Thread(target=TriggerActuatorToActivate, args=(tcp_ip, tcp_port), daemon=True)
    tcp_thread.start()

    # Continuously send data to the edge server via UDP
    while True:
        index = random.randrange(0, len(X))  # Adjust range to fit your DataFrame size
        data = X.values[index].tolist()
        Call_log( "Physical Layer", "ReadSensor", "Sensor read has been done")
        # Send the data to the edge server via UDP
        SendSensorDataToEdge(data, udp_ip, udp_port)
        Call_log( "Physical Layer", "SendSensorDataToEdge", "Sensor data has been sent to Edge")
        # Wait for 10 seconds before sending the next data
        time.sleep(10)

if __name__ == '__main__':
    main()

Physical server listening for modified data from edge server on 127.0.0.1:12348
Sent data to edge server via UDP: ['2/23/2022 11:00', 32.9, 23.9, 11.066, 6.35589, -1.95762, 15.3454, 9.6412, 347.9, 1001.72, 29.5807, 0.0, 80.8768, 702.129, 38.0512, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54656)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/6/2022 8:30', 32.9, 26.6, 25.9844, 20.8395, 22.1561, 3.87214, 3.23461, 115.5, 988.483, 29.1899, 0.19, 80.5671, 166.762, 48.3326, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/24/2022 9:30', 32.0, 28.4, 18.9266, 16.1231, 9.34916, 9.79107, 7.46689, 9.35, 992.105, 29.2968, 0.0, 88.5971, 463.635, 106.27, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54671)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge server via UDP: ['2/6/2

Connected to edge server at ('127.0.0.1', 54788)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge server via UDP: ['2/7/2022 1:00', 34.7, 26.6, 26.447, 24.8954, 22.7545, 3.58133, 3.1854, 210.7, 995.945, 29.4102, 0.19, 93.7558, 0.0, 0.0128392, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54792)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/23/2022 16:00', 32.0, 27.5, 15.854, 6.82588, 2.93936, 13.5178, 11.216, 25.85, 996.369, 29.4228, 0.0, 66.9188, 438.635, 145.097, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/3/2022 14:00', 32.9, 32.0, 16.826, 9.1944, 2.60762, 19.439, 14.0591, 354.6, 994.094, 29.3556, 0.0, 71.4042, 634.069, 166.677, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54801)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge s

Sent data to edge server via UDP: ['2/24/2022 21:00', 31.1, 24.8, 17.204, 13.2675, 11.1271, 5.04205, 3.90122, 359.5, 996.469, 29.4257, 0.0, 84.2084, 0.0, 0.0151033, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54868)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/2/2022 8:30', 39.2, 32.0, 25.36, 21.02, 12.98, 23.67, 15.27, 17.6, 987.23, 29.15, 0.03, 83.32, 153.27, 17.06, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/6/2022 23:30', 35.6, 28.4, 29.5142, 26.7493, 26.0365, 3.72674, 3.33304, 292.2, 995.246, 29.3896, 0.19, 89.2629, 0.0, 0.011859, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 54877)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge server via UDP: ['2/23/2022 11:00', 32.9, 23.9, 11.066, 6.35589, -1.95762, 15.3454, 9.6412, 347.9, 1001.72, 29.5807, 0.0, 8

Sent data to edge server via UDP: ['2/26/2022 11:00', 32.0, 60.8, 31.5842, 18.7678, nan, 3.21448, 1.81192, 84.4, 999.728, 29.5219, 0.0, 58.6292, 719.937, 620.22, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/7/2022 13:30', 48.2, 72.5, 54.608, 26.0107, nan, 14.1755, 8.80234, 234.8, 994.438, 29.3657, 0.01, 32.8863, 692.37, 653.983, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/25/2022 2:30', 30.2, 23.0, 11.714, 8.33675, 4.74805, 5.04205, 3.93253, 335.7, 999.52, 29.5158, 0.0, 85.9667, 0.0, 0.011652, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/27/2022 12:00', 43.7, 81.5, 50.504, 16.7462, nan, 9.20723, 5.22772, 324.9, 998.16, 29.4757, 0.22, 25.8195, 811.3, 779.007, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/25/2022 12:30', 31.1, 33.8, 25.9664, 11.9837, 17.2827, 11.5448, 8.31917, 18.25, 1003.78, 29.64

Sent data to edge server via UDP: ['2/23/2022 14:30', 32.0, 26.6, 14.198, 6.95049, -0.556384, 18.4771, 13.7057, 17.43, 997.728, 29.4629, 0.0, 72.3503, 690.488, 180.519, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 55079)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/2/2022 8:30', 39.2, 32.0, 25.36, 21.02, 12.98, 23.67, 15.27, 17.6, 987.23, 29.15, 0.03, 83.32, 153.27, 17.06, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/26/2022 12:00', 32.0, 72.5, 34.9844, 19.3611, nan, 3.1429, 1.52335, 169.8, 999.282, 29.5088, 0.0, 52.4718, 805.427, 754.208, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/28/2022 7:00', 32.9, 26.6, 24.728, 20.0799, nan, 2.92368, 1.87455, 162.0, 992.416, 29.306, 0.3, 82.1894, 0.0, 4.65612, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: [

Sent data to edge server via UDP: ['2/23/2022 3:00', 34.7, 27.5, 15.242, 4.77112, 2.66486, 15.2, 10.4398, 5.698, 999.753, 29.5227, 0.0, 62.5674, 0.0, 0.0116519, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/24/2022 21:30', 31.1, 23.9, 16.25, 12.4455, 11.284, 3.65292, 3.12053, 354.0, 996.86, 29.4373, 0.0, 84.6354, 0.0, 0.0144268, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 55227)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/3/2022 2:30', 33.8, 33.8, 15.73, 11.94, 0.86, 20.54, 14.76, 12.91, 992.42, 29.31, 0.03, 84.65, 0.0, 0.01, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 55231)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge server via UDP: ['2/22/2022 3:00', 41.0, 27.5, 24.3932, 16.2585, 9.9703, 28.4315, 20.1101, 334.3, 979.527, 28.9254, 0.0, 70.703, 0.0, 0

Sent data to edge server via UDP: ['2/5/2022 22:00', 33.8, 24.8, 33.4076, 19.8545, 27.3747, 8.76879, 6.73318, 144.2, 989.285, 29.2136, 0.19, 57.0744, 0.0, 0.0148133, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/7/2022 1:00', 34.7, 26.6, 26.447, 24.8954, 22.7545, 3.58133, 3.1854, 210.7, 995.945, 29.4102, 0.19, 93.7558, 0.0, 0.0128392, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 55305)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/25/2022 21:30', 32.0, 28.4, 26.8052, 13.3024, 18.9824, 9.71949, 7.34163, 64.46, 1000.32, 29.5394, 0.0, 56.1975, 0.0, 0.0122179, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/2/2022 18:30', 36.5, 31.1, 19.28, 16.06, 3.17, 27.18, 20.36, 14.42, 989.33, 29.22, 0.03, 87.03, 0.0, 0.02, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('

Sent data to edge server via UDP: ['2/28/2022 3:30', 33.8, 26.6, 29.4674, 20.1682, nan, 1.60836, 1.05583, 79.15, 992.502, 29.3086, 0.3, 67.8401, 0.0, 0.0126045, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/22/2022 10:30', 35.6, 42.8, 19.9706, 8.37812, 5.32385, 22.0786, 17.0208, 339.5, 990.61, 29.2527, 0.0, 60.1205, 628.777, 392.729, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/27/2022 8:30', 32.0, 32.9, 27.4676, 19.7417, nan, 2.55682, 0.565945, 293.7, 998.028, 29.4718, 0.19, 72.3105, 264.272, 270.445, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/23/2022 19:00', 32.0, 25.7, 16.016, 7.1001, 2.97402, 14.1039, 11.4956, 55.12, 995.636, 29.4011, 0.0, 67.282, 0.0, 0.0160973, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/6/2022 4:30', 32.9, 24.8, 29.426, 20.6419, 23.6292, 7.59887, 5.45812, 158.4, 986.883, 

Sent data to edge server via UDP: ['2/24/2022 23:00', 31.1, 23.9, 14.918, 11.6174, nan, 3.21448, 1.94613, 344.8, 998.177, 29.4762, 0.0, 86.461, 0.0, 0.0136261, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/4/2022 5:30', 32.9, 30.2, 12.83, 3.49649, 4.21779, 7.67269, 5.27917, 323.6, 1000.94, 29.5577, 0.0, 65.5902, 0.0, 0.0137228, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/5/2022 22:30', 33.8, 26.6, 33.9638, 19.9222, 26.5265, 13.4462, 9.27881, 152.9, 988.851, 29.2008, 0.19, 55.9791, 0.0, 0.011638, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/5/2022 15:30', 35.6, 57.2, 46.1516, 17.4688, 39.9424, 21.6983, 14.1576, 176.7, 990.898, 29.2612, 0.19, 31.3594, 440.967, 446.815, 'Yes', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/1/2022 15:00', 52.7, 66.2, 61.84, 48.96, nan, 15.05, 9.94, 144.6, 976.76, 28.84, 0.0

Sent data to edge server via UDP: ['2/24/2022 14:30', 32.0, 29.3, 21.7382, 18.0084, 11.7582, 12.4217, 8.8359, 341.0, 990.494, 29.2493, 0.0, 85.2752, 695.922, 146.91, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 56075)
Data received from edge server via TCP: triggerActuator 0
Sent data to edge server via UDP: ['2/23/2022 20:00', 32.0, 25.7, 16.034, 7.46083, 3.51973, 13.5178, 10.6433, 52.44, 995.598, 29.4, 0.0, 68.3388, 0.0, 0.0143855, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Sent data to edge server via UDP: ['2/25/2022 7:00', 29.3, 23.0, 12.488, 7.72114, nan, 2.1922, 1.51217, 354.8, 1002.53, 29.6048, 0.0, 80.7874, 0.0, 6.38113, 'No', 'Bermudagrass', '0.125"', 'Stilwlater, OK', nan]
Connected to edge server at ('127.0.0.1', 56085)
Data received from edge server via TCP: triggerActuator 1
Sent data to edge server via UDP: ['2/6/2022 18:00', 41.0, 37.4, 42.377, 28.589, 37.5719, 10.4487, 7.82033, 341.1, 991.544, 29.2803, 0.1

In [ ]:
# import socket
# import json
# import random
# import time

# def udp_client(data, ip, port):
#     # Create a UDP socket
#     client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

#     # Serialize the data to JSON format
#     message = json.dumps(data)

#     # Send data
#     client_socket.sendto(message.encode('utf-8'), (ip, port))
#     client_socket.close()

# # Server IP and port
# ip = '127.0.0.1'
# port = 12345
# # ip = 'localhost'
# # port= 10002
# # Example data to send (assuming X is a DataFrame)
# while True:
#     index = random.randrange(0, len(X))  # Adjust range to fit your DataFrame size
#     data = X.values[index].tolist()
    
#     # Send data to server
#     udp_client(data, ip, port)
    
#     print(f"Sent data: {data}")
    
#     # Wait for 10 seconds before sending the next data
#     time.sleep(10)



In [ ]:
df
